### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_failure_followup_survival",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5Z89R",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/519/heart+failure+clinical+records.zip && unzip heart+failure+clinical+records.zip && rm heart+failure+clinical+records.zip && mkdir -p local-data-warehouse/heart_failure_followup_survival && mv heart_failure_clinical_records_dataset.csv local-data-warehouse/heart_failure_followup_survival/
""",
    # References
    academic_reference_bibtex="""@article{chicco2020machine,
  title={Machine learning can predict survival of patients with heart failure from serum creatinine and ejection fraction alone},
  author={Chicco, Davide and Jurman, Giuseppe},
  journal={BMC medical informatics and decision making},
  volume={20},
  number={1},
  pages={16},
  year={2020},
  publisher={Springer}
}
""",
    academic_reference_bibtex_key="chicco2020machine",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use the data as is from UCI.

- We keep all features of the dataset. The study that introduced the dataset also curated a subset of features. We leave it to the pipeline to select the relevant features for the task.
- There is a "time" feature in the data which is the follow-up time in days. The authors used this in some part of the experiments and showed it improved the predictive performance. We also keep this feature and follow their preprocessing to transform it into months (from days). Note, this does not make the data temporal, as the split is across unseen patients (not across time) and we only have one entry per patient.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DEATH_EVENT",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="DEATH_EVENT",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "heart_failure_clinical_records_dataset.csv")
print("Loaded data shape:", df.shape)

as_cat_type = ["anaemia", "DEATH_EVENT","diabetes", "high_blood_pressure", "sex", "smoking"]
df[as_cat_type] = df[as_cat_type].astype("category")

# Transform time from days to months
df["month_to_follow_up"] = df["time"] / 30.0
df = df.drop(columns=["time"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (299, 13)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 299
Columns: 13
Use sampling: False (sample size: 299)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['creatinine_phosphokinase', 'platelets', 'month_to_follow_up', 'age', 'serum_creatinine', 'serum_sodium', 'ejection_fraction', 'smoking', 'diabetes', 'anaemia']
Rows remaining as candidates after top-10 filter: 0 (of 299)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,DEATH_EVENT,month_to_follow_up
0,70.0,0,582,0,40,0,51000.0,2.7,136,1,1,0,8.333333
1,50.0,1,298,0,35,0,362000.0,0.9,140,1,1,0,8.000000
2,45.0,0,2442,1,30,0,334000.0,1.1,139,1,0,1,4.300000
3,80.0,1,123,0,35,1,388000.0,9.4,133,1,1,1,0.333333
4,42.0,0,102,1,40,0,237000.0,1.2,140,1,0,0,2.466667


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,anaemia,category,0.0,0.0,2.0,"0, 1"
1,diabetes,category,0.0,0.0,2.0,"0, 1"
2,high_blood_pressure,category,0.0,0.0,2.0,"0, 1"
3,sex,category,0.0,0.0,2.0,"1, 0"
4,smoking,category,0.0,0.0,2.0,"0, 1"
5,DEATH_EVENT,category,0.0,0.0,2.0,"0, 1"
6,age,float64,0.0,0.0,47.0,"60.0, 50.0, 65.0, 70.0, 45.0, 55.0, 75.0, 53.0, 58.0, 63.0"
7,platelets,float64,0.0,0.0,176.0,"263358.03, 237000.0, 235000.0, 255000.0, 271000.0, 228000.0, 279000.0, 221000.0, 226000.0, 305000.0"
8,serum_creatinine,float64,0.0,0.0,40.0,"1.0, 0.9, 1.1, 1.2, 0.8, 1.3, 0.7, 1.18, 1.7, 1.4"
9,month_to_follow_up,float64,0.0,0.0,148.0,"8.3333, 6.2333, 6.2, 0.3333, 3.5667, 2.9, 8.1333, 6.9667, 7.1333, 1.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,299.0,60.833893,11.894809,40.000000,95.0
creatinine_phosphokinase,299.0,581.839465,970.287881,23.000000,7861.0
ejection_fraction,299.0,38.083612,11.834841,14.000000,80.0
platelets,299.0,263358.029264,97804.236869,25100.000000,850000.0
serum_creatinine,299.0,1.393880,1.034510,0.500000,9.4
serum_sodium,299.0,136.625418,4.412477,113.000000,148.0
month_to_follow_up,299.0,4.342029,2.587140,0.133333,9.5


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column              rank                    
DEATH_EVENT         1        0    203  67.89
                    2        1     96  32.11
anaemia             1        0    170  56.86
                    2        1    129  43.14
diabetes            1        0    174  58.19
                    2        1    125  41.81
high_blood_pressure 1        0    194  64.88
                    2        1    105  35.12
sex                 1        1    194  64.88
                    2        0    105  35.12
smoking             1        0    203  67.89
                    2        1     96  32.11

In [8]:
# Target Distribution
target_df

,count,pct
DEATH_EVENT,,
0,203,67.89
1,96,32.11


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7bff-a1d3-788d-8b8d-86bad1c6fb5f
9227a8fee35987d5f64482258b1465fc4174ddecc34745371d49b3be1a8be60c
